In [ ]:
import uuid
import pandas as pd
import psycopg2
import sys
import os
from qdrant_client import QdrantClient
from qdrant_client.http import models
from qdrant_client.models import Distance, VectorParams, SparseVectorParams
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
sys.path.append(os.path.abspath(os.path.join('..')))
from data_ingestion.connect_to_google_drive import get_sheets_client, SHEET_ID
from fastembed import SparseTextEmbedding

c:\Users\Olena\Downloads\ucu-employee-support-rag\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Olena\Downloads\ucu-employee-support-rag\.venv\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.7) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [ ]:
QDRANT_HOST = "localhost"
QDRANT_PORT = 6333

In [ ]:
client = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)

In [ ]:
sheets_client = get_sheets_client()
sheet = sheets_client.open_by_key(SHEET_ID).sheet1
df = pd.DataFrame(sheet.get_all_records())
to_sync = df[(df['status'] == 'Success') & (df['vector_db_sync'] == 'No')]

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

In [ ]:
def get_text_from_postgres(file_id):
    try:
        conn = psycopg2.connect(
            dbname="ucu_rag_db", user="user", password="password", host="127.0.0.1", port=5432
        )
        cur = conn.cursor()
        cur.execute(
            "SELECT markdown_content FROM processed_documents WHERE google_drive_id = %s", 
            (file_id,)
        )
        result = cur.fetchone()
        cur.close()
        conn.close()
        return result[0] if result else None
    except Exception as e:
        print(f"ERROR: {e}")
        return None

In [ ]:
def upload_to_qdrant(to_sync, model, collection, chunking_method, e5=False, sparse_model=None):
    for _, row in to_sync.iterrows():
            file_id = row['google_drive_id']
            file_name = row['file_name']
            
            text = get_text_from_postgres(file_id)

            if text:
                if chunking_method == "fixed_size":
                    chunks = [text[i:i+1000] for i in range(0, len(text), 800)]
                elif chunking_method == "recursive_character":
                    chunks = text_splitter.split_text(text)
                for i, chunk in enumerate(chunks):
                    if e5:
                        dense_vector = model.encode("passage: " + chunk).tolist()
                    else:
                        dense_vector = model.encode(chunk).tolist()
                    vectors_to_upload = {"default": dense_vector}
                    if sparse_model:
                        sparse_emb = list(sparse_model.embed([chunk]))[0]
                        vectors_to_upload["text_sparse"] = models.SparseVector(
                            indices=sparse_emb.indices.tolist(),
                            values=sparse_emb.values.tolist()
                        )
                    point_id = str(uuid.uuid5(uuid.NAMESPACE_DNS, f"{file_id}_{i}"))
                    client.upsert(
                        collection_name=collection,
                        points=[
                            models.PointStruct(
                                id=point_id,
                                vector=vectors_to_upload,
                                payload={
                                    "file_id": file_id,
                                    "file_name": file_name,
                                    "text": chunk,
                                    "chunk_index": i
                                }
                            )
                        ]
                    )
                print(f"File {file_name} uploaded into Qdrant")

<hr>

# paraphrase-multilingual-MiniLM-L12-v2

In [8]:
MODEL_BASELINE = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2728.96it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
COLLECTION_BASELINE = "ucu_documents_baseline"

In [10]:
if not client.collection_exists(COLLECTION_BASELINE):
    client.create_collection(
        collection_name=COLLECTION_BASELINE,
        vectors_config={
            "default": VectorParams(
                size=384,
                distance=Distance.COSINE
            )
        }
    )

In [11]:
upload_to_qdrant(to_sync=to_sync, model=MODEL_BASELINE, collection=COLLECTION_BASELINE, chunking_method="fixed_size")

File Polityka-pidtrymky-simyi-ta-simejnyh-tsinnostej..pdf uploaded into Qdrant
File Polityka-pidtrymky-veteraniv-i-veteranok.pdf uploaded into Qdrant
File Політика співпраці з жертводавцями _ Відділ розвитку УКУ.pdf uploaded into Qdrant
File Polityka-skarg-i-propozytsij-ta-poryadok-vyrishennya-konfliktnyh-sytuatsij-v-UKU-1.pdf uploaded into Qdrant
File Protydiya-nasylstvu.pdf uploaded into Qdrant
File Polityka-shhodo-zapobigannya-ta-zahystu-vid-seksualnoyi-ekspluatatsiyi-ta-nasylstva-1.pdf uploaded into Qdrant
File Polityka-pro-ekologichnu-ta-sotsialnu-stijkist-UKU.pdf uploaded into Qdrant
File Polityka-zahystu-vrazlyvyh-osib-UKU-2020.pdf uploaded into Qdrant
File Polityka-UKU-shhodo-zahodiv-za-uchasti-politychnyh-diyachiv-v-UKU-2-21.pdf uploaded into Qdrant
File pamyatka-pro-zasady-duhovnoyi-formatsiyi-v-uku-1-2.pdf uploaded into Qdrant
File Antykoruptsijna-polityka-UKU.pdf uploaded into Qdrant
File 4-pamyatka-pro-etos-uku.pdf uploaded into Qdrant
File 3-polozhennya-pro-duhovnu-gromad

In [12]:
COLLECTION_BASELINE_RECURSIVE_CHAR = "ucu_documents_baseline_recursive_char"

In [13]:
if not client.collection_exists(COLLECTION_BASELINE_RECURSIVE_CHAR):
    client.create_collection(
        collection_name=COLLECTION_BASELINE_RECURSIVE_CHAR,
        vectors_config={
            "default": VectorParams(
                size=384,
                distance=Distance.COSINE
            )
        }
    )

In [19]:
upload_to_qdrant(to_sync=to_sync, model=MODEL_BASELINE, collection=COLLECTION_BASELINE_RECURSIVE_CHAR, chunking_method="recursive_character")

File Polityka-pidtrymky-simyi-ta-simejnyh-tsinnostej..pdf uploaded into Qdrant
File Polityka-pidtrymky-veteraniv-i-veteranok.pdf uploaded into Qdrant
File Політика співпраці з жертводавцями _ Відділ розвитку УКУ.pdf uploaded into Qdrant
File Polityka-skarg-i-propozytsij-ta-poryadok-vyrishennya-konfliktnyh-sytuatsij-v-UKU-1.pdf uploaded into Qdrant
File Protydiya-nasylstvu.pdf uploaded into Qdrant
File Polityka-shhodo-zapobigannya-ta-zahystu-vid-seksualnoyi-ekspluatatsiyi-ta-nasylstva-1.pdf uploaded into Qdrant
File Polityka-pro-ekologichnu-ta-sotsialnu-stijkist-UKU.pdf uploaded into Qdrant
File Polityka-zahystu-vrazlyvyh-osib-UKU-2020.pdf uploaded into Qdrant
File Polityka-UKU-shhodo-zahodiv-za-uchasti-politychnyh-diyachiv-v-UKU-2-21.pdf uploaded into Qdrant
File pamyatka-pro-zasady-duhovnoyi-formatsiyi-v-uku-1-2.pdf uploaded into Qdrant
File Antykoruptsijna-polityka-UKU.pdf uploaded into Qdrant
File 4-pamyatka-pro-etos-uku.pdf uploaded into Qdrant
File 3-polozhennya-pro-duhovnu-gromad

<hr>

## Add sparse vectors for hybrid search

In [ ]:
MODEL_SPARSE = SparseTextEmbedding(model_name="Qdrant/bm25")

In [13]:
COLLECTION_BASELINE_HYBRID = "ucu_documents_baseline_hybrid"

In [14]:
if not client.collection_exists(COLLECTION_BASELINE_HYBRID):
    client.create_collection(
        collection_name=COLLECTION_BASELINE_HYBRID,
        vectors_config={
            "default": VectorParams(
                size=384,
                distance=Distance.COSINE
            )
        },
        sparse_vectors_config={
            "text_sparse": SparseVectorParams(
                index=models.SparseIndexParams(
                    on_disk=False,
                )
            )
        }
    )

In [15]:
upload_to_qdrant(to_sync=to_sync, model=MODEL_BASELINE, collection=COLLECTION_BASELINE_HYBRID, chunking_method="recursive_character", sparse_model=MODEL_SPARSE)

File Polityka-pidtrymky-simyi-ta-simejnyh-tsinnostej..pdf uploaded into Qdrant
File Polityka-pidtrymky-veteraniv-i-veteranok.pdf uploaded into Qdrant
File Політика співпраці з жертводавцями _ Відділ розвитку УКУ.pdf uploaded into Qdrant
File Polityka-skarg-i-propozytsij-ta-poryadok-vyrishennya-konfliktnyh-sytuatsij-v-UKU-1.pdf uploaded into Qdrant
File Protydiya-nasylstvu.pdf uploaded into Qdrant
File Polityka-shhodo-zapobigannya-ta-zahystu-vid-seksualnoyi-ekspluatatsiyi-ta-nasylstva-1.pdf uploaded into Qdrant
File Polityka-pro-ekologichnu-ta-sotsialnu-stijkist-UKU.pdf uploaded into Qdrant
File Polityka-zahystu-vrazlyvyh-osib-UKU-2020.pdf uploaded into Qdrant
File Polityka-UKU-shhodo-zahodiv-za-uchasti-politychnyh-diyachiv-v-UKU-2-21.pdf uploaded into Qdrant
File pamyatka-pro-zasady-duhovnoyi-formatsiyi-v-uku-1-2.pdf uploaded into Qdrant
File Antykoruptsijna-polityka-UKU.pdf uploaded into Qdrant
File 4-pamyatka-pro-etos-uku.pdf uploaded into Qdrant
File 3-polozhennya-pro-duhovnu-gromad

<hr>

# intfloat/multilingual-e5-small

In [16]:
MODEL_E5_SMALL = SentenceTransformer('intfloat/multilingual-e5-small')
COLLECTION_E5_SMALL = "ucu_documents_e5_small"

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2894.53it/s]
BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [17]:
if not client.collection_exists(COLLECTION_E5_SMALL):
    client.create_collection(
        collection_name=COLLECTION_E5_SMALL,
        vectors_config={
            "default": VectorParams(
                size=384,
                distance=Distance.COSINE
            )
        },
        sparse_vectors_config={
            "text_sparse": SparseVectorParams(
                index=models.SparseIndexParams(
                    on_disk=False,
                )
            )
        }
    )

In [18]:
upload_to_qdrant(to_sync=to_sync, model=MODEL_E5_SMALL, collection=COLLECTION_E5_SMALL, chunking_method="recursive_character", e5=True, sparse_model=MODEL_SPARSE)

File Polityka-pidtrymky-simyi-ta-simejnyh-tsinnostej..pdf uploaded into Qdrant
File Polityka-pidtrymky-veteraniv-i-veteranok.pdf uploaded into Qdrant
File Політика співпраці з жертводавцями _ Відділ розвитку УКУ.pdf uploaded into Qdrant
File Polityka-skarg-i-propozytsij-ta-poryadok-vyrishennya-konfliktnyh-sytuatsij-v-UKU-1.pdf uploaded into Qdrant
File Protydiya-nasylstvu.pdf uploaded into Qdrant
File Polityka-shhodo-zapobigannya-ta-zahystu-vid-seksualnoyi-ekspluatatsiyi-ta-nasylstva-1.pdf uploaded into Qdrant
File Polityka-pro-ekologichnu-ta-sotsialnu-stijkist-UKU.pdf uploaded into Qdrant
File Polityka-zahystu-vrazlyvyh-osib-UKU-2020.pdf uploaded into Qdrant
File Polityka-UKU-shhodo-zahodiv-za-uchasti-politychnyh-diyachiv-v-UKU-2-21.pdf uploaded into Qdrant
File pamyatka-pro-zasady-duhovnoyi-formatsiyi-v-uku-1-2.pdf uploaded into Qdrant
File Antykoruptsijna-polityka-UKU.pdf uploaded into Qdrant
File 4-pamyatka-pro-etos-uku.pdf uploaded into Qdrant
File 3-polozhennya-pro-duhovnu-gromad

<hr>

# intfloat/multilingual-e5-base

In [9]:
MODEL_E5_BASE = SentenceTransformer('intfloat/multilingual-e5-base')
COLLECTION_E5_BASE = "ucu_documents_e5_base"

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2924.11it/s]
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
if not client.collection_exists(COLLECTION_E5_BASE):
    client.create_collection(
        collection_name=COLLECTION_E5_BASE,
        vectors_config={
            "default": VectorParams(
                size=768,
                distance=Distance.COSINE
            )
        },
        sparse_vectors_config={
            "text_sparse": SparseVectorParams(
                index=models.SparseIndexParams(
                    on_disk=False,
                )
            )
        }
    )

In [13]:
upload_to_qdrant(to_sync=to_sync, model=MODEL_E5_BASE, collection=COLLECTION_E5_BASE, chunking_method="recursive_character", e5=True, sparse_model=MODEL_SPARSE)

File plagiarism_check_policy.pdf uploaded into Qdrant
File academic_integrity_policy.pdf uploaded into Qdrant
File Plan-rozvytku-polityky-rivnosti-ta-inklyuzyvnogo-robochogo-seredovyshhe-v-UKU.pdf uploaded into Qdrant
File Polozhennya-pro-Otsinku-efektyvnosti-pratsivnykiv-Ukrayinskogo-katolytskogo-Universytetu-1.pdf uploaded into Qdrant
File Polozhennya-pro-pidbir-pratsivnykiv-Ukrayinskogo-katolytskogo-universytetu-1.pdf uploaded into Qdrant
File Pravyla-vnutrishnogo-trudovogo-rozporyadku-1.pdf uploaded into Qdrant
File Polozhennya-pro-sluzhbovi-vidryadzhennya-personalu-ZVO-UKU.pdf uploaded into Qdrant
File Polozhennya-pro-dystantsijnu-robotu-ta-gnuchkyj-rezhym-robochogo-chasu-Ukrayinskogo-katolytskogo-universytetu-1.pdf uploaded into Qdrant
File Polozhennya-pro-shhorichnu-vidznaku-Vykladach-roku.pdf uploaded into Qdrant
File Poryadok-realizatsiyi-programy-rozvytku-molodyh-naukovo-pedagogichnyh-ta-pedagogichnyh-pratsivnykiv-UKU.pdf uploaded into Qdrant
File Polozhennya-pro-poryadok-vyz

<hr>

# intfloat/multilingual-e5-large

In [8]:
MODEL_E5_LARGE = SentenceTransformer('intfloat/multilingual-e5-large')
COLLECTION_E5_LARGE = "ucu_documents_e5_large"

c:\Users\Olena\Downloads\ucu-employee-support-rag\.venv\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Olena\.cache\huggingface\hub\models--intfloat--multilingual-e5-large. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 1840.07it/s]
XLMRobertaMod

In [9]:
if not client.collection_exists(COLLECTION_E5_LARGE):
    client.create_collection(
        collection_name=COLLECTION_E5_LARGE,
        vectors_config={
            "default": VectorParams(
                size=1024,
                distance=Distance.COSINE
            )
        },
        sparse_vectors_config={
            "text_sparse": SparseVectorParams(
                index=models.SparseIndexParams(
                    on_disk=False,
                )
            )
        }
    )

In [12]:
upload_to_qdrant(to_sync=to_sync, model=MODEL_E5_LARGE, collection=COLLECTION_E5_LARGE, chunking_method="recursive_character", e5=True, sparse_model=MODEL_SPARSE)

File Polityka-pidtrymky-simyi-ta-simejnyh-tsinnostej..pdf uploaded into Qdrant
File Polityka-pidtrymky-veteraniv-i-veteranok.pdf uploaded into Qdrant
File Політика співпраці з жертводавцями _ Відділ розвитку УКУ.pdf uploaded into Qdrant
File Polityka-skarg-i-propozytsij-ta-poryadok-vyrishennya-konfliktnyh-sytuatsij-v-UKU-1.pdf uploaded into Qdrant
File Protydiya-nasylstvu.pdf uploaded into Qdrant
File Polityka-shhodo-zapobigannya-ta-zahystu-vid-seksualnoyi-ekspluatatsiyi-ta-nasylstva-1.pdf uploaded into Qdrant
File Polityka-pro-ekologichnu-ta-sotsialnu-stijkist-UKU.pdf uploaded into Qdrant
File Polityka-zahystu-vrazlyvyh-osib-UKU-2020.pdf uploaded into Qdrant
File Polityka-UKU-shhodo-zahodiv-za-uchasti-politychnyh-diyachiv-v-UKU-2-21.pdf uploaded into Qdrant
File pamyatka-pro-zasady-duhovnoyi-formatsiyi-v-uku-1-2.pdf uploaded into Qdrant
File Antykoruptsijna-polityka-UKU.pdf uploaded into Qdrant
File 4-pamyatka-pro-etos-uku.pdf uploaded into Qdrant
File 3-polozhennya-pro-duhovnu-gromad

<HR>

# lang-uk/ukr-paraphrase-multilingual-mpnet-base

In [8]:
MODEL_UKR = SentenceTransformer('lang-uk/ukr-paraphrase-multilingual-mpnet-base')
COLLECTION_UKR = "ucu_documents_ukr"

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11994.92it/s]
XLMRobertaModel LOAD REPORT from: lang-uk/ukr-paraphrase-multilingual-mpnet-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
if not client.collection_exists(COLLECTION_UKR):
    client.create_collection(
        collection_name=COLLECTION_UKR,
        vectors_config={
            "default": VectorParams(
                size=768,
                distance=Distance.COSINE
            )
        },
        sparse_vectors_config={
            "text_sparse": SparseVectorParams(
                index=models.SparseIndexParams(
                    on_disk=False,
                )
            )
        }
    )

In [12]:
upload_to_qdrant(to_sync=to_sync, model=MODEL_UKR, collection=COLLECTION_UKR, chunking_method="recursive_character", sparse_model=MODEL_SPARSE)

File Polityka-pidtrymky-simyi-ta-simejnyh-tsinnostej..pdf uploaded into Qdrant
File Polityka-pidtrymky-veteraniv-i-veteranok.pdf uploaded into Qdrant
File Політика співпраці з жертводавцями _ Відділ розвитку УКУ.pdf uploaded into Qdrant
File Polityka-skarg-i-propozytsij-ta-poryadok-vyrishennya-konfliktnyh-sytuatsij-v-UKU-1.pdf uploaded into Qdrant
File Protydiya-nasylstvu.pdf uploaded into Qdrant
File Polityka-shhodo-zapobigannya-ta-zahystu-vid-seksualnoyi-ekspluatatsiyi-ta-nasylstva-1.pdf uploaded into Qdrant
File Polityka-pro-ekologichnu-ta-sotsialnu-stijkist-UKU.pdf uploaded into Qdrant
File Polityka-zahystu-vrazlyvyh-osib-UKU-2020.pdf uploaded into Qdrant
File Polityka-UKU-shhodo-zahodiv-za-uchasti-politychnyh-diyachiv-v-UKU-2-21.pdf uploaded into Qdrant
File pamyatka-pro-zasady-duhovnoyi-formatsiyi-v-uku-1-2.pdf uploaded into Qdrant
File Antykoruptsijna-polityka-UKU.pdf uploaded into Qdrant
File 4-pamyatka-pro-etos-uku.pdf uploaded into Qdrant
File 3-polozhennya-pro-duhovnu-gromad

<hr>